# <font color='#2E86C1' size=6>**Missing values in practice, and predicting new purchases**</font>

**Workflow:** [01 Data cleaning](01_data_cleaning.ipynb) → [02 Customer table](02_customer_table.ipynb) → [03 Missing values](03_missing_values.ipynb) → [04 Outliers](04_outlier_detection.ipynb) → [05 Pipeline and models](05_pipeline_and_models.ipynb) → [06 Feature engineering and selection](06_feature_engineering_selection.ipynb) → [07 Classification modelling](07_classification_modelling.ipynb) → **08 Missing values and deployment**

Week 10 of the course, applied to our pipeline. Notebook 03 chose *how* each gap is filled; this one asks four practical questions about the **numeric fallback imputer** (what fills the gaps the history fill and the formulas leave), then saves the models and predicts new purchases from the terminal:

| Question | Section | Challenge |
|---|---|---|
| Where are the gaps left, and can we just delete them? | 1 | |
| Which imputers can go into a pipeline that predicts new purchases? (the label trap) | 2 | |
| Does the imputer change the model's score? | 3 | A1 |
| Which imputer guesses the hidden values best, and does seeing the categories help? | 4 | B2 |
| What does MICE add when it fills each gap several times? | 5 | B1 |
| Random-forest proximities from scratch: better guesses? usable in `predict.py`? | 6 | B3 |
| Saving the model and predicting new purchases from the terminal | 7 | A2, B4 |

The slow comparisons (sections 3, 4 and 6: about 15 minutes) come from
`python -m src.experiments --experiment imputers --output-dir docs/experiments/imputers`; this notebook reads their tables and runs the quick parts live. Development set only: the locked test set is never touched.

# <font color='#2E86C1' size=5>**0. Setup**</font>

In [1]:
import os
import sys
import warnings
from pathlib import Path

# run from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import yaml

import predict
from src.data import load_data
from src.evaluate import temporal_folds
from src.experiments import imputer_recipes, mask_numeric_values
from src.history import CustomerHistoryFiller
from src.pipeline import load_pipeline, versions_path
from src.preprocessing import _NUMERIC_IMPUTERS, build_preprocessor
from src.tasks import TASKS
from src.pipeline import prepare_task_data

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
with open("config.yaml", encoding="utf-8") as file:
    CONFIG = yaml.safe_load(file)
EXPERIMENTS = Path("docs/experiments/imputers")

data = prepare_task_data(CONFIG, "classification")
recipe = data["preprocessing_config"]
X_dev, y_dev = data["X_dev"], data["y_dev"]
numeric, categorical = recipe["numeric_features"], recipe["categorical_features"]
folds = temporal_folds(data["extras_dev"][CONFIG["data"]["date_column"]], CONFIG["cv"]["n_splits"], CONFIG["cv"]["gap_days"])
print(f"classification development set: {len(X_dev)} rows, {len(numeric)} numeric and {len(categorical)} categorical features")
print("numeric imputers in the pipeline (imputation.numeric_strategy):", list(_NUMERIC_IMPUTERS))

classification development set: 5024 rows, 23 numeric and 3 categorical features
numeric imputers in the pipeline (imputation.numeric_strategy): ['median', 'mean', 'constant', 'knn', 'iterative', 'random_forest']


# <font color='#2E86C1' size=5>**1. Where are the gaps left, and can we just delete them?**</font>

Most gaps never reach an imputer: notebook 01 recovers values from identities and notebook 03 fills purchase gaps from the customer's own earlier purchases (`CustomerHistoryFiller`, the first step of the pipeline). What is left after that is the imputers' job. (Here the history step is fitted on all development rows, for a description only; in CV it is refitted per fold.)

In [2]:
history = CustomerHistoryFiller(recipe["customer_history"]).fit(X_dev)
filled = history.transform(X_dev)
rows = []
for column in numeric + categorical:
    before, after = X_dev[column].isna(), filled[column].isna()
    if before.any():
        rows.append({"column": column, "kind": "numeric" if column in numeric else "categorical",
                     "missing": int(before.sum()), "after history fill": int(after.sum()),
                     "% after": round(100 * after.mean(), 1),
                     "repurchase | missing": y_dev[after].mean() if after.any() else np.nan,
                     "repurchase | present": y_dev[~after].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False), "\n")

complete = ~filled[numeric + categorical].isna().any(axis=1)
print(f"deleting every row with a gap keeps {complete.sum()} of {len(filled)} purchases ({complete.mean():.0%})")
print(f"repurchase rate: all {y_dev.mean():.3f}  ·  complete rows {y_dev[complete].mean():.3f}")
kaggle = data["X_kaggle"][numeric + categorical].isna().any(axis=1)
print(f"test.csv: {kaggle.sum()} of {len(kaggle)} purchases have at least one gap before the history fill -- each needs a prediction")

                      column        kind  missing  after history fill  % after  repurchase | missing  repurchase | present
days_since_previous_purchase     numeric        6                   0      0.0                   NaN                 0.550
    prior_max_purchase_value     numeric       36                  36      0.7                 0.500                 0.550
         prior_mean_gap_days     numeric        6                   0      0.0                   NaN                 0.550
                  spend_365d     numeric      138                 138      2.7                 0.565                 0.549
                payment_type categorical      144                  18      0.4                 0.278                 0.551
                 salesperson categorical      158                  25      0.5                 0.360                 0.551
            transport_method categorical      153                  22      0.4                 0.318                 0.551 

deleting every

* **Few gaps reach the imputers.** After the history fill, only `spend_365d` (about 3%) and `prior_max_purchase_value` (under 1%) still have numeric gaps, plus a few categories. Everything else was recovered exactly earlier in the pipeline.
* **Deleting rows isn't an option** for a deployed model: a new purchase with a gap still needs a prediction, and test.csv has gaps too. So the pipeline must know how to fill them anyway.
* **A gap can carry information.** `spend_365d` keeps its `MissingIndicator` flag (`mnar_indicator_sources`): after the identity fill it is missing only for customers with more than 365 days of tenure. The few categorical gaps left go with a much lower repurchase rate (28–36% vs 55%), but on only ~20 rows each. Treating them as a `"Missing"` category instead of the mode was compared in notebook 03 §15 ([experiment record](../docs/preprocessing_experiments.md)). These are categorical gaps; this notebook is about the numeric imputer.
* **Deleting** every row with a gap would keep 95% of the development rows with almost the same repurchase rate. It wouldn't bias much, but it doesn't solve deployment either.

# <font color='#2E86C1' size=5>**2. The label trap**</font>

The theory classes also fill gaps **with the label**: mean / mode by class, and random-forest proximities from a forest grown on the label. Two practical problems:

* **In cross-validation** they leak: a validation purchase's gap filled with the mean of *its own* class carries the answer. Anything that learns from the label must be fitted on the training rows of each fold.
* **At prediction time there is no label.** A gap of a new purchase can't be filled "by class". The theory's workaround for random forests is one copy of the new row per class, each imputed and predicted.

So `_NUMERIC_IMPUTERS` holds only imputers that learn from the **features**: median, mean, constant, KNN, MICE (`iterative`) and missForest (`random_forest`, an `IterativeImputer` whose regressor is a random forest). They run unchanged in CV and on new purchases. Section 6 builds the proximity imputer anyway, to see what the label buys.

# <font color='#2E86C1' size=5>**3. Does the imputer change the model's score? (A1, B1, B2)**</font>

Each task's configured model (logistic regression; random forest averaged over 3 seeds per fold), the same 5 temporal folds, only the numeric fallback imputer changed. Row limits, `fit_rows` and the regression's `spend_365d` formula stay, as in the real pipeline. Every row is compared with the median on the same folds:

* `mean_advantage`, `folds_a_better` / `folds_tied`: the descriptive paired comparison used throughout the project (positive = better than the median, for RMSE too);
* `corrected_t`, `corrected_p`: the course's corrected paired t-test (Nadeau & Bengio), with the mean validation/training ratio of our folds. It is an **approximation** here: the correction assumes random resampling, and expanding temporal windows overlap even more. A large p-value means "no evidence of a difference", not "equivalent".

In [3]:
columns = ["variant", "mean_score", "fold_std", "mean_advantage", "folds_a_better", "folds_tied",
           "corrected_t", "corrected_p", "seconds_per_cv"]
for task in TASKS:
    summary = pd.read_csv(EXPERIMENTS / f"imputers_{task}_summary.csv")
    print(f"{task} ({summary['metric'].iloc[0]}, {int(summary['seeds'].iloc[0])} model seed(s) per fold)")
    print(summary[columns].to_string(index=False, float_format=lambda v: f"{v:.4f}"), "\n")

classification (accuracy, 1 model seed(s) per fold)
                 variant  mean_score  fold_std  mean_advantage  folds_a_better  folds_tied  corrected_t  corrected_p  seconds_per_cv
                  median      0.7095    0.0168          0.0000               0           5       0.0000       1.0000          4.9423
                     knn      0.7109    0.0161          0.0014               3           2       0.8727       0.4321          6.2713
               iterative      0.7097    0.0174          0.0002               3           0       0.2185       0.8377          7.5500
           random_forest      0.7100    0.0175          0.0005               3           0       0.3566       0.7394         46.7982
          knn+categories      0.7107    0.0162          0.0012               3           2       0.9757       0.3845          5.5576
    iterative+categories      0.7105    0.0166          0.0010               4           1       2.1376       0.0994         19.7940
random_forest+cat

* **No imputer changes the model.** Classification: the largest gain over the median is +0.0014 accuracy (KNN: 3 folds better, 2 tied), about one purchase per validation fold of ~840. Regression: the largest is 0.64 euros of RMSE out of 1,938 (`iterative x5`, better on all 5 folds but 11 times slower). Every corrected p-value is above 0.09, so there is no evidence of a difference.
* **Seeing the categories (B2)** doesn't help either: within ±0.4 euros and ±0.0005 accuracy of the same imputer without them.
* **Cost:** the median takes ~5 s per cross-validation. missForest takes 40–50 s, and 80–100 s with the categories.
* **Decision: keep the median** (`numeric_strategy: median`). Section 4 explains why better guesses don't show up in the score.

# <font color='#2E86C1' size=5>**4. Which imputer guesses best? (and B2: letting it see the categories)**</font>

A real gap can't be checked. So we **make** gaps where we know the answer: in the validation rows of every fold, hide 200 observed values of each column that has real gaps (among purchases with history: the gaps' own population), fit each imputer on the training rows (history fill → imputer → row limits, exactly the pipeline's numeric step) and measure how far each guess lands from the truth: mean absolute error, in days or euros. The `+categories` variants also see the categorical columns, one-hot encoded (`imputation.categorical_helpers: true`).

In [4]:
errors = pd.read_csv(EXPERIMENTS / "imputers_classification_hidden_values_summary.csv")
table = errors.pivot(index="variant", columns="column", values="mean").loc[
    lambda t: t.index.isin(errors["variant"].unique())]
order = [v for v in ["median", "knn", "knn+categories", "iterative", "iterative+categories",
                     "random_forest", "random_forest+categories", "rf_proximity (B3)"] if v in table.index]
print("mean absolute error of the guesses, mean of 5 folds (lower is better):")
print(table.loc[order].round(1).to_string())

mean absolute error of the guesses, mean of 5 folds (lower is better):
column                    days_since_previous_purchase  prior_max_purchase_value  prior_mean_gap_days  spend_365d
variant                                                                                                          
median                                            91.7                    1153.9                 65.1      2355.9
knn                                               49.4                     519.3                 26.0      1348.2
knn+categories                                    55.8                     497.8                 29.5      1358.2
iterative                                        123.9                     440.8                 72.9      1270.1
iterative+categories                             123.1                     430.4                 73.3      1262.5
random_forest                                     24.2                     362.6                 21.8      1216.7
random_forest+cat

* **missForest (`random_forest`) guesses best on every column**: about half the median's error on the spend and maximum columns, and a quarter on the day columns. KNN is close on the day columns; MICE (`iterative`) is close on the spend columns.
* **Seeing the categories barely changes the guesses** (within ±10%), unlike the course's `age_cat`, which halved the error on `age`. A model-based imputer is only as good as the columns it can see, and payment type, salesperson and transport method say little about a customer's spending history. The purchase-history columns it already sees are the informative ones.
* **So why doesn't the model improve?** Three reasons. Few values are imputed (≈3% of one column, <1% of another). The row limits already pull a poor median guess back into a plausible range. And the `spend_365d` missing flag tells the model that the value was guessed. A much better guess for 3% of one column moves no prediction across the threshold.

# <font color='#2E86C1' size=5>**5. MICE as the theory describes it: several completed datasets (B1)**</font>

`IterativeImputer()` gives **one** completed dataset: each gap is the regression's best guess. The theory's MICE draws each value at random around that guess (`sample_posterior=True`), repeats with another seed to get *m* datasets, trains a model on each and combines them. In our pipeline: `imputation.numeric_strategy: iterative` + `n_imputations: 5` (`MultipleImputationClassifier` / `...Regressor` in `src/pipeline.py`).

First, what the draws look like for one hidden `spend_365d`:

In [5]:
train, validation = folds[-1]
history = CustomerHistoryFiller(recipe["customer_history"]).fit(X_dev.iloc[train])
train_filled, validation_filled = history.transform(X_dev.iloc[train]), history.transform(X_dev.iloc[validation])
with_history = np.flatnonzero(validation_filled["prior_purchase_count"].gt(0).to_numpy())
masked, truth = mask_numeric_values(validation_filled, ["spend_365d"], with_history, 5, seed=3)
person = truth["spend_365d"].index[0]

def guess(sample_posterior, seed):
    candidate = imputer_recipes(recipe, ["iterative"])["iterative"]
    candidate["imputation"].update(sample_posterior=sample_posterior, random_state=seed)
    step = (build_preprocessor(candidate).fit(train_filled, y_dev.iloc[train])
            .named_transformers_["numeric"].named_steps["impute"])
    return step.transform(masked.loc[[person], numeric])[0, numeric.index("spend_365d")]

draws = [guess(True, seed) for seed in range(5)]
print(f"purchase {person}: true spend_365d {truth['spend_365d'].iloc[0]:,.0f} euros")
print(f"best guess (one completed dataset): {guess(False, 0):,.0f}")
print(f"5 completed datasets drew: {', '.join(f'{d:,.0f}' for d in draws)}")

purchase 1147: true spend_365d 2,480 euros


best guess (one completed dataset): 974
5 completed datasets drew: 1,970, 1,665, 734, 734, 734


* The *m* datasets disagree about the hidden value: that spread **is** the imputation uncertainty a single imputation hides. (After drawing, each value is still kept inside the row's limits.)
* In our pipeline the draws happen in **training** only. Once a copy is fitted, it fills a new purchase's gaps with its own best guess. Otherwise a fitted imputer keeps drawing, and the same purchase would get a different prediction on every call, depending on the other rows in the batch (a test covers this). The copies still disagree on a new row, because each learned from a different completed dataset.

The scores are in section 3, as `iterative x5`:

* In the demo, the true value is 2,480 euros and the best guess is 974. The five draws spread from 734 to 1,970: three of them fall below the row's lower limit (from its spend windows and smallest purchase) and are raised to it, 734. This is the uncertainty a single imputation hides.
* `iterative x5` is the only variant better than the median on **all five** regression folds, but by 0.64 euros of RMSE at about 11 times the cost; classification: +0.0010 accuracy, 3 folds better and 2 worse. **Not worth five times the models** while so few values are imputed. It stays available as `n_imputations` for a dataset with more gaps.

# <font color='#2E86C1' size=5>**6. Random-forest proximities, from scratch (B3)**</font>

`src/experiments.py` → `proximity_impute`: a random forest is grown on the **label** (repurchase) from the training rows, with their gaps filled by the training medians. `forest.apply(X)` gives each row's leaf in every tree, and two rows' proximity is the share of trees where they share a leaf. A hidden value is the proximity-weighted mean of the training rows where the column is observed. The validation rows' own labels are never used: they are median-filled and only pushed down the trees. Its errors are the `rf_proximity (B3)` row of section 4's table:

In [6]:
print(table.loc[[v for v in ["median", "random_forest", "rf_proximity (B3)"] if v in table.index]].round(1).to_string())

column             days_since_previous_purchase  prior_max_purchase_value  prior_mean_gap_days  spend_365d
variant                                                                                                   
median                                     91.7                    1153.9                 65.1      2355.9
random_forest                              24.2                     362.6                 21.8      1216.7
rf_proximity (B3)                          61.9                     923.8                 44.3      2493.6


* **The proximity imputer beats the median** on three of the four columns (e.g. 62 vs 92 days on `days_since_previous_purchase`). It is **worse** on `spend_365d` (2,494 vs 2,356 euros), and far behind missForest everywhere (e.g. 924 vs 363 euros on `prior_max_purchase_value`). The forest decides who is "close" by what predicts *repurchase*. That is not what predicts the missing column, so the neighbours it finds are the wrong ones for imputing.
* **Could it run in `predict.py`?** Technically yes, with median-filled queries as here: a new row only needs to be pushed down the trees, so its label is never needed. But the model would have to keep the forest **and the whole training matrix** (it grows with the data), and the forest needs the target to grow. It can be fitted inside a pipeline (y is passed to `fit`), never on unlabelled data. The theory's class-wise initial fill would need one copy of every new row per class. Worse guesses, more memory and a label dependency: **not used.**

# <font color='#2E86C1' size=5>**7. Saving the model and predicting new purchases (A2, B4)**</font>

`python main.py` refits each task's pipeline on all development rows and **saves** it: `models/<task>_pipeline.joblib`, plus `models/<task>_pipeline.versions.json` with the library versions it was saved with. The file holds the **whole fitted Pipeline**: the frozen purchase history, the imputers' learned values, the encoders, the scaler and the model. A new purchase goes through exactly the same steps with those values. Nothing is re-learned from new data.

`predict.py` loads the saved pipelines and predicts a CSV with the columns of test.csv, with no training. `data/examples/new_purchases.csv` has 20 purchases from test.csv, made messy on purpose:

In [7]:
tokens = CONFIG["diagnostics"]["placeholder_tokens"]
new = pd.read_csv("data/examples/new_purchases.csv", dtype=str, keep_default_na=False)
shown = [3, 7, 11, 12, 14, 15, 16, 18, 19]
new.loc[shown, ["ID", "customer_id", "Custo", "days_since_previous_purchase", "payment_type", "salesperson",
                "transport_method", "spend_365d"]]

,ID,customer_id,Custo,days_since_previous_purchase,payment_type,salesperson,transport_method,spend_365d
3,10002081,531725,?,38.0,Maybe Tomorrow Payment 2,VLOOKUP Master 1,Maybe Tomorrow Delivery 56,24198.550000000003
7,10004094,738735,1809.3075886000006,,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Maybe Tomorrow Delivery 43,11142.65
11,10002835,795676,368.5216364999997,127.0,Maybe Tomorrow Payment 99,VLOOKUP Master 1,Maybe Tomorrow Delivery 56,6928.710000000001
12,10006795,652441,3214.9343467000012,298.0,Maybe Tomorrow Payment 1,X,Maybe Tomorrow Delivery 56,1355.63
14,10003596,962551,377.03619030000004,118.0,Maybe Tomorrow Payment 7,VLOOKUP Master 9,Maybe Tomorrow Delivery 61,4176.360000000001
15,10003596,962551,377.03619030000004,118.0,Maybe Tomorrow Payment 7,VLOOKUP Master 9,Maybe Tomorrow Delivery 61,4176.360000000001
16,10001016,218902,268.36276710000004,42.0,Maybe Tomorrow Payment 7,VLOOKUP Master 1,N/A,7189.1900000000005
18,10006552,999999,120.4542685,35.0,Maybe Tomorrow Payment 7,,Maybe Tomorrow Delivery 61,4868.64
19,10004706,206888,82.35716400000001,77.0,Maybe Tomorrow Payment 12,VLOOKUP Master 9,Maybe Tomorrow Delivery 61,1000000000


<div style="background-color:#fff8e1; border-left:6px solid #f9a825; padding:10px 14px; border-radius:4px">🤔 <b>Guess first:</b> row 3 has <code>Custo = "?"</code>, row 7 a blank <code>days_since_previous_purchase</code>, row 11 a payment type never seen in training, row 12 a salesperson <code>"X"</code> that isn't a label, row 16 <code>"N/A"</code>, row 18 a customer with no earlier purchase on file, row 19 the placeholder <code>1000000000</code>, and rows 14–15 are the same purchase twice. How many predictions should come out?</div>

In [8]:
pipelines = {task: load_pipeline(predict.model_path(CONFIG, task)) for task in TASKS}
clientes = load_data(CONFIG["data"]["clientes_path"], tokens)
predictions = predict.predict(load_data("data/examples/new_purchases.csv", tokens), pipelines, CONFIG, clientes)
print(f"{len(new)} rows in, {len(predictions)} rows out")
predictions.loc[shown]

20 rows in, 20 rows out


,ID,repeat_probability,repeat_purchase_90d,spend_next_90d
3,10002081,0.759418,1,5660.31
7,10004094,0.550957,1,2609.90
11,10002835,0.454504,0,2219.47
12,10006795,0.274778,0,2382.84
14,10003596,0.591567,1,1003.84
15,10003596,0.591567,1,1003.84
16,10001016,0.860928,1,1482.82
18,10006552,0.778491,1,1783.24
19,10004706,0.305647,0,491.65


* Rows 3, 7, 16 and 19: the cleaning turns `"?"`, blanks, `"N/A"` and the numeric placeholder into gaps; the pipeline's saved history step and imputers fill them.
* Rows 11 and 12: a valid but unseen label is kept, and the target encoder gives it the training mean; `"X"` fails the schema pattern and becomes a gap, then the categorical imputer fills it. Nothing crashes.
* Row 18: no earlier purchase to borrow from, so the fitted imputers take over.
* Rows 14 and 15: same purchase, same prediction, **both kept**. `prepare_table` never drops a row: one prediction per input row, in the input's order.
* `spend_next_90d` is the spend **if** the customer returns: the regression only learned from returning customers.

**B4: a safer `predict.py`.** It stops with a clear message when the input lacks a column the model needs, and warns when the installed library versions differ from the saved ones:

In [9]:
broken = load_data("data/examples/new_purchases.csv", tokens).drop(columns=["Custo", "payment_type"])
try:
    predict.predict(broken, pipelines, CONFIG, clientes)
except ValueError as error:
    print("ValueError:", error)
print("\nsaved with:", json.loads(versions_path(predict.model_path(CONFIG, "classification")).read_text()))

ValueError: The input lacks 2 column(s) the model needs: ['cost', 'payment_type']. It must have the columns of test.csv (original or English names).

saved with: {'python': '3.12.10', 'scikit-learn': '1.7.2', 'xgboost': '3.4.1', 'numpy': '2.5.4', 'pandas': '3.0.6', 'scipy': '1.18.1', 'joblib': '1.6.0', 'category-encoders': '2.11.1'}


From the project folder (Windows, macOS and Linux):

```bash
python main.py                                                     # train, evaluate, save models/<task>_pipeline.joblib
python predict.py data/examples/new_purchases.csv predictions.csv  # predict both tasks: 20 rows
python predict.py data/raw/test.csv predictions.csv --task classification
```

Output columns: `ID`, `repeat_probability`, `repeat_purchase_90d` (at 0.5) and `spend_next_90d`. Tests check that rows out = rows in, that duplicates are kept, that `predict.py` on test.csv matches the pipeline on the prepared test matrix, the missing-column message and the version warning (`tests/test_pipeline.py`).

<div style="background-color:#fdecea; border-left:6px solid #c62828; padding:10px 14px; border-radius:4px">⚠️ Load a model with the <b>same library versions</b> it was saved with, and only load files you trust: a <code>.joblib</code> file can run code when it is loaded.</div>

# <font color='#2E86C1' size=5>**8. Summary**</font>

* **Deleting** isn't an option for a deployed model: every new purchase with a gap still needs a prediction.
* Gaps that carry information (`spend_365d`) get a **flag** as well as a filled value.
* **The label trap:** methods that fill gaps with the label leak in CV and can't run on unlabelled data as they are. Our imputers learn from the features only.
* The imputer is now a **setting** (`numeric_strategy`: median, mean, constant, knn, iterative, random_forest). missForest guesses hidden values best, but **no imputer changes either model's score**: the earlier exact recoveries leave too few gaps. We **keep the median**: as good, 10 times faster, simplest.
* A model-based imputer is **only as good as the columns it sees**. Our categories add nothing to the guesses.
* **MICE** in full = several completed datasets → several models → combined. It is implemented (`n_imputations`), but it isn't worth its cost here.
* **Deployment:** `main.py` saves each pipeline with its library versions. `predict.py` predicts any test.csv-like file from the terminal, one row per input row, and refuses inputs that lack a needed column.

| File | Change |
|---|---|
| `src/preprocessing.py` | `_NUMERIC_IMPUTERS` registry (median, mean, constant, knn, iterative, **random_forest**); `RowBoundedImputer` can also see the categorical columns (`helper_features`); `build_preprocessor` passes them when `categorical_helpers: true` |
| `src/pipeline.py` | `MultipleImputationClassifier` / `...Regressor` (`n_imputations`); `imputer_label`; `save_pipeline` writes the library versions and `load_pipeline` checks them; `check_input_columns`, `prepare_new_records` |
| `src/evaluate.py` | `corrected_paired_test`, `fold_test_train_ratio` |
| `src/tuning.py` | search spaces also reach the model inside multiple imputation (`pipeline__model__C`) |
| `src/experiments.py` | `--experiment imputers`: model scores per imputer, hidden-value errors, `proximity_impute` (B3) |
| `main.py`, `src/results.py` | the imputer is printed, recorded in `runs.csv` and in the report header |
| `predict.py` (new) | predicts a CSV from the terminal, one row per input row |
| `config.yaml` | `imputation`: `random_forest` option, `max_iter`, `n_estimators`, `categorical_helpers`, `helper_min_frequency`, `n_imputations` |
| `data/examples/new_purchases.csv`, `.gitignore` | the messy example; `predictions.csv` ignored (`models/` already was) |